Establish connection with Bronze Delta Lake table, define a specific simulated business day

In [ ]:
import os
import datetime
import deltalake
import polars as pl

# Create the DeltaTable object
table = deltalake.DeltaTable(
    "s3://lakehouse/bronze/raw_events",
        storage_options={
        "AWS_ENDPOINT_URL": "http://[IP of MinIO]:9000",
        "AWS_REGION": "local",
        "AWS_S3_FORCE_PATH_STYLE": "true",
        "AWS_ALLOW_HTTP": "true",
    },
)

# Load ONLY the Bronze table on a specific day
arrow_table = table.to_pyarrow_table(
    filters=[("business_date", "=", datetime.date(2026, 3, 2))]
)

df = pl.from_arrow(arrow_table)


Show count of total events by business date

In [ ]:
df.group_by("business_date").count()

Show count of total events, grouped by type of event

In [ ]:
df.group_by("event_type").count()


First 10 records

In [ ]:
df.head(10)

Total records count

In [ ]:
df.height

Sums up the total rows across all Parquet files in the partitioned section of the total Delta Lake Bronze table based on simulated business date

In [ ]:
import pyarrow.parquet as pq
import pyarrow.fs as fs
from urllib.parse import urlparse

s3 = fs.S3FileSystem(
    endpoint_override="http://[IP of MinIO]:9000",
    region="local",
)

files_31 = [f for f in table.file_uris() if "2026-03-02" in f]

counts = []

for uri in files_31:
    parsed = urlparse(uri)
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")
    full_path = f"{bucket}/{key}"
    pa_tbl = pq.read_table(full_path, filesystem=s3)
    counts.append((uri, pa_tbl.num_rows))

print("\nRow counts per file:")
for f, n in counts:
    print(f"{n:>8} rows  |  {f}")

print("\nTOTAL ROWS:", sum(n for _, n in counts))


Latest snapshotted commit for the global Delta Lake Bronze table. See the latest JSON file lakehouse/bronze/raw_events/_delta_log/00000000000000000###.json

In [ ]:
table.version()

In [ ]:
import polars as pl

hist = table.history()       
pl.DataFrame(hist[:10]) 


Last 5 checkpoint offset values that Spark uses to keep track of which messages it's already processed in the Kafka message queue. See MinIO location lakehouse/checkpoints/bronze_raw_events/main/offsets. Download the file, open with TextEdit to see the JSON where it shows exactly which Kafka offsets that Spark has committed most recently.

In [ ]:
import pyarrow.fs as fs

s3 = fs.S3FileSystem(
    endpoint_override="http://[IP of MinIO]:9000",
    region="local",
)

# List all offset files
files = s3.get_file_info(
    fs.FileSelector("lakehouse/checkpoints/bronze_raw_events/offsets")
)

# Filter out directories and sort by filename
offset_files = sorted(
    [f for f in files if f.type == fs.FileType.File],
    key=lambda f: int(f.base_name)
)

offset_files[-5:]


Statistics related to purchase events. So, average books sold in a single transaction

In [ ]:
df.filter(pl.col("event_type") == "purchase") \
  .select(
      pl.col("items")
        .list.eval(pl.element().struct.field("quantity"))
        .list.sum()
        .alias("units_sold")
  ).describe()